# Narkomfin Building Graph Analysis - PART 1

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print(Helper.Version())

The version that you are using (0.9.43) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:

In [3]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Load both floor plans

In [5]:
from pathlib import Path

HERE = Path.cwd()  # VS Code sets CWD to the notebook's folder

BREP_PLAN1 = HERE / 'output' / 'narkomfin_plan1_face.brep'
BREP_PLAN2 = HERE / 'output' / 'narkomfin_plan2_face.brep'

plan1 = Topology.ByBREPPath(str(BREP_PLAN1))
plan2 = Topology.ByBREPPath(str(BREP_PLAN2))
print('Both plans loaded.')


Both plans loaded.


## 6. Show the geometry

In [6]:
Topology.Show(plan1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='white',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(plan2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='white',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 7. Create grid overlays for both plans

In [7]:
# ── Plan 1 ─────────────────────────────────────────────────────────────────
b_r1    = Wire.BoundingRectangle(plan1)
d1      = Topology.Dictionary(b_r1)
xmin1   = Dictionary.ValueAtKey(d1, 'xmin')
xmax1   = Dictionary.ValueAtKey(d1, 'xmax')
ymin1   = Dictionary.ValueAtKey(d1, 'ymin')
ymax1   = Dictionary.ValueAtKey(d1, 'ymax')
width1  = Dictionary.ValueAtKey(d1, 'width')
length1 = Dictionary.ValueAtKey(d1, 'length')
uRange1 = list(range(0, int(width1)+2, 2))
vRange1 = list(range(0, int(length1)+2, 2))
grid1   = Grid.EdgesByDistances(plan1, clip=True, uRange=uRange1, vRange=vRange1)

# ── Plan 2 ─────────────────────────────────────────────────────────────────
b_r2    = Wire.BoundingRectangle(plan2)
d2      = Topology.Dictionary(b_r2)
xmin2   = Dictionary.ValueAtKey(d2, 'xmin')
xmax2   = Dictionary.ValueAtKey(d2, 'xmax')
ymin2   = Dictionary.ValueAtKey(d2, 'ymin')
ymax2   = Dictionary.ValueAtKey(d2, 'ymax')
width2  = Dictionary.ValueAtKey(d2, 'width')
length2 = Dictionary.ValueAtKey(d2, 'length')
uRange2 = list(range(0, int(width2)+2, 2))
vRange2 = list(range(0, int(length2)+2, 2))
grid2   = Grid.EdgesByDistances(plan2, clip=True, uRange=uRange2, vRange=vRange2)


## 8. Show the geometry and the grids

In [8]:
Topology.Show(plan1, grid1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='grey',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(plan2, grid2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='grey',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 9. Slice each floor plan with its grid to create topologic shells

In [9]:
# ── Plan 1 ─────────────────────────────────────────────────────────────────
shell1 = Topology.Slice(plan1, grid1)
faces1 = Topology.Faces(shell1)
for i, f in enumerate(faces1):
    d = Dictionary.ByKeyValue('face_id', 'p1_face_'+str(i+1))
    f = Topology.SetDictionary(f, d)

# ── Plan 2 ─────────────────────────────────────────────────────────────────
shell2 = Topology.Slice(plan2, grid2)
faces2 = Topology.Faces(shell2)
for i, f in enumerate(faces2):
    d = Dictionary.ByKeyValue('face_id', 'p2_face_'+str(i+1))
    f = Topology.SetDictionary(f, d)

print(f'Shell 1: {len(faces1)} faces   Shell 2: {len(faces2)} faces')


Shell 1: 269 faces   Shell 2: 312 faces


## 10. Show both shells

In [10]:
Topology.Show(shell1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor='black',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(shell2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor='black',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 11. Derive navigation and analysis graphs from both shells

In [11]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they come from
navigation_graph1 = Graph.ByTopology(shell1, direct=False, viaSharedTopologies=True)
analysis_graph1   = Graph.ByTopology(shell1)

navigation_graph2 = Graph.ByTopology(shell2, direct=False, viaSharedTopologies=True)
analysis_graph2   = Graph.ByTopology(shell2)


## 12. Derive and store the analysis graph vertices

In [12]:
g_verts1 = Graph.Vertices(analysis_graph1)
g_verts2 = Graph.Vertices(analysis_graph2)


## 13. Show both analysis graphs

In [13]:
Topology.Show(analysis_graph1,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(analysis_graph2,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


## 14. Spatial Intelligence through Graph Analysis

### b. Shortest Path (Use navigation graph)

In [15]:
import time

# ── Plan 1 ─────────────────────────────────────────────────────────────────
start1 = Vertex.ByCoordinates(xmin1+2, ymax1-2, 0)
end1   = Vertex.ByCoordinates(xmax1-2, ymin1+2, 0)
crg1   = Graph.CompiledRoutingGraph(navigation_graph1, precomputeTurns=False)
t0     = time.time()
shortest_path1 = Graph.ShortestPath(crg1, vertexA=start1, vertexB=end1)
print('Plan 1 — Shortest Path:', round(time.time()-t0, 2), 's')
straight_path1 = Wire.Straighten(shortest_path1, host=plan1)
print('  Original length:', round(Wire.Length(shortest_path1), 2))
print('  Straightened length:', round(Wire.Length(straight_path1), 2))
for edge in Topology.Edges(shortest_path1):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'red']))
for edge in Topology.Edges(straight_path1):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'blue']))

# ── Plan 2 ─────────────────────────────────────────────────────────────────
start2 = Vertex.ByCoordinates(xmin2+2, ymax2-2, 0)
end2   = Vertex.ByCoordinates(xmax2-2, ymin2+2, 0)
crg2   = Graph.CompiledRoutingGraph(navigation_graph2, precomputeTurns=False)
t0     = time.time()
shortest_path2 = Graph.ShortestPath(crg2, vertexA=start2, vertexB=end2)
print('Plan 2 — Shortest Path:', round(time.time()-t0, 2), 's')
straight_path2 = Wire.Straighten(shortest_path2, host=plan2)
print('  Original length:', round(Wire.Length(shortest_path2), 2))
print('  Straightened length:', round(Wire.Length(straight_path2), 2))
for edge in Topology.Edges(shortest_path2):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'red']))
for edge in Topology.Edges(straight_path2):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'blue']))


Plan 1 — Shortest Path: 0.17 s
  Original length: 77.18
  Straightened length: 74.4
Plan 2 — Shortest Path: 0.44 s
  Original length: 130.31
  Straightened length: 107.6


In [16]:
Topology.Show(plan1, shortest_path1, straight_path1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey='color',
              edgeWidthKey='width',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(plan2, shortest_path2, straight_path2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey='color',
              edgeWidthKey='width',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [17]:
centrality_list1 = Graph.ClosenessCentrality(analysis_graph1, colorScale='thermal')
centrality_list2 = Graph.ClosenessCentrality(analysis_graph2, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [18]:
reset_dictionaries(shell1)
faces1 = Topology.Faces(shell1)
_ = transfer_dicts_by_key(faces1, g_verts1, 'face_id')

reset_dictionaries(shell2)
faces2 = Topology.Faces(shell2)
_ = transfer_dicts_by_key(faces2, g_verts2, 'face_id')


In [19]:
Topology.Show(faces1,
              faceColorKey='cc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(faces2,
              faceColorKey='cc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [20]:
centrality_list1 = Graph.BetweennessCentrality(analysis_graph1, normalize=True, colorScale='thermal')
centrality_list2 = Graph.BetweennessCentrality(analysis_graph2, normalize=True, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [21]:
reset_dictionaries(shell1)
faces1 = Topology.Faces(shell1)
_ = transfer_dicts_by_key(faces1, g_verts1, 'face_id')

reset_dictionaries(shell2)
faces2 = Topology.Faces(shell2)
_ = transfer_dicts_by_key(faces2, g_verts2, 'face_id')


In [22]:
Topology.Show(faces1,
              faceColorKey='bc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

Topology.Show(faces2,
              faceColorKey='bc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)
